# P05 and P06 · cathodic vs anodic, 30 Hz burst and ARC-EX

**The question.** The usual montage puts the **cathode** over the spine. Reversing it to **anodic**
changes which fibres see the steepest voltage gradient. So: **does the arm respond the same to
cathodic and anodic stimulation, is the answer the same for 30 Hz and for ARC-EX, and does it
replicate between two participants?**

**Two participants, one session each, the same 2 x 2 x 2.** Polarity (**cathodic** / **anodic**)
x lidocaine (**before** / **with**) x protocol (**30 Hz burst** / **ARC-EX**), electrode 2, anode
at the iliac crests, sixteen recordings in all.

| | | |
|---|---|---|
| **P05** (`NTA`) | 24-07-2026 | log writes polarity as *cathodic / anodic* |
| **P06** | 28-09-2026 | log writes it as *negative / positive* — negative is cathodic, which its own note on a negative row (*"erreur (cathodique)"*) confirms |

Each participant is a **panel per block**, so the four panels are P05 before, P05 with lidocaine,
P06 before, P06 with lidocaine. Reading across panels within a participant shows what lidocaine
did; reading the same panel position down the figure shows whether the two people agree.

**Two things are measured.**

1. **The threshold moves** — the anodic threshold as a % of the cathodic one, same muscle, same
   protocol, same block. This uses the thresholds you picked, nothing else.
2. **The response at a matched intensity** — anodic read at the **cathodic threshold**, as a % of
   cathodic there. Same current, same muscle, same minute: the only thing that differs is which
   way round the field is.

**The rule for intensity.** Every muscle is analysed **at its own motor threshold**, picked by
hand in section 2 — never at one mA for the whole arm, and never a step above. Within a protocol
and a block, cathodic and anodic are read at the **same** intensity (the cathodic threshold), or
the bars would compare current instead of polarity. 30 Hz and ARC-EX are never forced onto one
intensity, and neither are the two participants.

**Millivolts never travel between participants** — different skin, electrodes and gain. The mV
panel in section 4c is to be read **within** a participant only; everything else on this page is a
percentage of that participant's own control, which is what makes the two comparable at all.

**Colours:** 30 Hz burst **grey**, ARC-EX **red**; **plain = cathodic, hatched = anodic**.

In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from functions import (set_style, load_run, pretty, thresholds_for, threshold_source,
                       condition_table, fig_vibration, fig_bars,
                       threshold_picker, save_threshold_csv, load_threshold_csv, mt_file,
                       mt_stamp, warn_if_stale,
                       fig_threshold_grid, fig_detection_grid)
set_style()
from functions.paper import motor_thresholds

## 1 · Config

In [ ]:
# ======== the recordings ========
D05 = "tSCS_CHUV_data/24-07-2026/testSCS/"   # P05, 24-07-2026, electrode 2
D06 = "tSCS_CHUV_data/28-08-2026/"           # P06, 28-09-2026 (folder is misnamed), electrode 2

RUN = {   # (panel, condition) -> csv.   panel = "<participant> · <block>"
  ("P05 · before",    "30 Hz · cathodic"):  D05 + "Burst_autosave_20260724_102443_670ms.csv",
  ("P05 · before",    "30 Hz · anodic"):    D05 + "Burst_autosave_20260724_102721_694ms.csv",
  ("P05 · before",    "ARC-EX · cathodic"): D05 + "Modulated_autosave_20260724_103530_508ms.csv",
  ("P05 · before",    "ARC-EX · anodic"):   D05 + "Modulated_autosave_20260724_103849_459ms.csv",
  ("P05 · lidocaine", "30 Hz · cathodic"):  D05 + "Burst_autosave_20260724_113834_522ms.csv",
  ("P05 · lidocaine", "30 Hz · anodic"):    D05 + "Burst_autosave_20260724_114055_891ms.csv",
  ("P05 · lidocaine", "ARC-EX · cathodic"): D05 + "Modulated_autosave_20260724_114332_473ms.csv",
  ("P05 · lidocaine", "ARC-EX · anodic"):   D05 + "Modulated_autosave_20260724_114730_561ms.csv",
  ("P06 · before",    "30 Hz · cathodic"):  D06 + "Burst_autosave_20260928_152710_593ms.csv",
  ("P06 · before",    "30 Hz · anodic"):    D06 + "Burst_autosave_20260928_152456_242ms.csv",
  ("P06 · before",    "ARC-EX · cathodic"): D06 + "Modulated_autosave_20260928_154023_427ms.csv",
  ("P06 · before",    "ARC-EX · anodic"):   D06 + "Modulated_autosave_20260928_153621_340ms.csv",
  ("P06 · lidocaine", "30 Hz · cathodic"):  D06 + "Burst_autosave_20260928_165756_941ms.csv",
  ("P06 · lidocaine", "30 Hz · anodic"):    D06 + "Burst_autosave_20260928_165525_067ms.csv",
  ("P06 · lidocaine", "ARC-EX · cathodic"): D06 + "Modulated_autosave_20260928_170517_985ms.csv",
  ("P06 · lidocaine", "ARC-EX · anodic"):   D06 + "Modulated_autosave_20260928_170249_644ms.csv",
}
SUBJECTS   = ["P05", "P06"]
BLOCKS     = ["before", "lidocaine"]
PANELS     = [f"{s} · {b}" for s in SUBJECTS for b in BLOCKS]   # one panel each, in this order
PROTOCOLS  = ["30 Hz", "ARC-EX"]
POLARITIES = ["cathodic", "anodic"]
CONDITIONS = [f"{p} · {pol}" for p in PROTOCOLS for pol in POLARITIES]

# which protocol each condition belongs to: conditions in different groups are never compared
# and never share an intensity
GROUP = {f"{p} · {pol}": p for p in PROTOCOLS for pol in POLARITIES}
# ...and each one is read as a % of its OWN protocol's CATHODIC recording in the same panel
BASE  = {f"{p} · {pol}": f"{p} · cathodic" for p in PROTOCOLS for pol in POLARITIES}

# colour = protocol only; polarity is the fill pattern - cathodic plain, anodic hatched
COLOUR = {"30 Hz · cathodic": "#9A9A9A", "30 Hz · anodic": "#9A9A9A",
          "ARC-EX · cathodic": "#C0392B", "ARC-EX · anodic": "#C0392B"}
HATCH  = {"30 Hz · cathodic": "",        "30 Hz · anodic": "///",
          "ARC-EX · cathodic": "",       "ARC-EX · anodic": "///"}
PROTO_COLOUR = {"30 Hz": "#9A9A9A", "ARC-EX": "#C0392B"}

# the same four muscles as every other across-participants notebook
MUSCLES = ["Flex. digitorum (L)", "Flex. digitorum (R)", "Ext. digitorum (L)",
           "Flex. carpi rad. (R)"]

KW = dict(n_pulses=10, resp_start_ms=8.0, resp_end_ms=None, guard_ms=1.0, min_snr=1.2,
          max_edge_frac=0.5, snr_on="median", anchor="mean", anchor_win_ms=3.0)

runs = RUN
for pan in PANELS:
    for cond in CONDITIONS:
        f = runs[(pan, cond)]
        amps = [m["amp_ma"] for m in load_run(f)[0]]
        print(f"{pan:17s} {cond:20s} {f.split('/')[-1][-22:]}  {amps[0]:g}-{amps[-1]:g} mA, "
              f"step {amps[1] - amps[0]:g}")

## 2 · The motor thresholds you chose

Read straight from `results/<session>/mt_*.csv`, so this is what you
actually picked — nothing here can substitute a different value. One row per recording, one column
per muscle: every intensity stacked, **your threshold in red**, the mA in the corner.
`not picked` = nothing saved for that muscle in that recording; change it in section 5.

Each **cathodic** block sets the intensity its protocol is analysed at in that block. The
**anodic** picks are the first result in their own right — how far the threshold moves when the
field is reversed — and are read in section 4a.

In [ ]:
CHECK = {f"{pan} · {cond}": runs[(pan, cond)] for pan in PANELS for cond in CONDITIONS}

picked = fig_threshold_grid(CHECK, MUSCLES,
                            title="P05 and P06 — the thresholds saved for each recording")

### The numbers, and where each one came from

In [ ]:
# hand-picked thresholds win wherever they exist; detection fills in the rest. Pick them in
# section 5 or in notebooks/motor_thresholds/ and re-run this cell - the source is printed below.
MT = {}      # MT[(panel, protocol)] = {muscle: mA}, from that panel's CATHODIC recording
for pan in PANELS:
    for p in PROTOCOLS:
        csv = runs[(pan, f"{p} · cathodic")]
        MT[(pan, p)] = thresholds_for(csv, MUSCLES, consecutive=2, **KW)
        print(f"{pan:17s} {p:7s} cathodic   {threshold_source(csv)}")

MT_ALL = {lab: picked.get(lab, {}) for lab in CHECK}      # every recording's own picks

print("\nmotor threshold (mA), as picked in each recording")
for pan in PANELS:
    print(f"\n  {pan}")
    print(f"  {'muscle':22s}" + "".join(f"{c.replace(' · ', ' '):>20s}" for c in CONDITIONS))
    for m in MUSCLES:
        print(f"  {m:22s}" + "".join(f"{str(MT_ALL[f'{pan} · {c}'].get(m) or '-'):>20s}"
                                     for c in CONDITIONS))

MT_STAMP = mt_stamp(runs)      # so the figures can tell you if a pick is saved after this

## 3 · Are the peaks detected correctly? — every recording, every muscle

The same grid as section 2 — a row per recording, a column per muscle — but showing how the peaks
are found at the threshold picked in **that** recording. The 10 pulses are re-aligned on their own
onset, with the max (v) and min (^) the peak-to-peak is made of.

| | |
|---|---|
| **green** | the response window — `RESP_START_MS` after the pulse to `GUARD_MS` before the next |
| **red bands** | the only places a peak may be taken: `ANCHOR_WIN_MS` either side of the train's own average latency. **A peak outside these is skipped however clear it looks** |
| markers stacked | the same deflection found on every pulse |
| red ring | flagged — on a window border, or at a different latency from the rest |
| corner | `clean`, or how many of the 10 pulses were flagged |

An anodic block whose train sits in the noise is a *result*, not a detection failure — check here
which of the two it is before believing a small bar in section 4.

### If a peak you can see is not being taken

`TRY` changes the settings **for this figure only**, so you can test whether detection is too
strict without touching any result. When you find settings you want, copy them into `KW` in
section 1.

| setting | now | loosen it by | what it controls |
|---|---|---|---|
| `anchor_win_ms` | 3.0 | **raising** it, or `anchor=None` | how far from the train's average latency a peak may sit — much the most common reason a visible response is missed |
| `resp_start_ms` | 8.0 | **lowering** it | when the window opens after the pulse; too low and it takes the stimulus artifact |
| `min_snr` | 1.2 | **lowering** it, or `None` | how far above that muscle's noise a train must be to count at all |
| `max_edge_frac` | 0.5 | **raising** it, or `None` | how many pulses may peak on a window border before the whole train is called artifact |

In [ ]:
# ======== try different detection settings, for this figure only ========
TRY = dict(KW)                     # start from the analysis settings...
# TRY["anchor_win_ms"] = 6.0       # ...and loosen one at a time: widen the red bands
# TRY["anchor"] = None             # no anchoring: biggest max/min anywhere in the window
# TRY["resp_start_ms"] = 6.0       # open the window sooner after the pulse
# TRY["min_snr"] = None            # stop discarding trains that sit close to noise
# TRY["max_edge_frac"] = None      # stop discarding trains that peak on a window border
# ========================================================================

fig_detection_grid(CHECK, MT_ALL, MUSCLES, edge_ms=1.0, jitter_ms=0.5,
                   title="P05 and P06 and P06 — peak detection at each recording's own threshold", **TRY)

## 4 · The results

### 4a · How far the threshold moves when the field is reversed

The **anodic** threshold as a % of the **cathodic** one, same muscle, same protocol, same block.
**100 % = polarity costs nothing.** Above 100 % means anodic needed more current; below means it
needed less. Grey is 30 Hz, red is ARC-EX — if one protocol is much more sensitive to polarity
than the other, that is what this figure shows.

The raw milliamps are printed underneath, because a % of a 35 mA threshold and a % of a 105 mA one
are not the same thing in current.

In [ ]:
shift = {}
for pan in PANELS:
    for p in PROTOCOLS:
        cat = MT_ALL[f"{pan} · {p} · cathodic"]
        ano = MT_ALL[f"{pan} · {p} · anodic"]
        for m in MUSCLES:
            if cat.get(m) and ano.get(m):
                shift[(pan, p, m)] = dict(pct=100 * ano[m] / cat[m], cat=cat[m], ano=ano[m])

fig_vibration(shift, PANELS, MUSCLES, PROTOCOLS, colours=PROTO_COLOUR,
              ylabel="% of cathodic", ref=100, ylim=(0, 150),
              titles={pan: pan for pan in PANELS},
              title="P05 and P06 — anodic motor threshold, % of cathodic")

print(f"{'panel':18s}{'muscle':22s}" + "".join(f"{p + ' cath':>15s}{p + ' anod':>15s}{'%':>7s}"
                                               for p in PROTOCOLS))
for pan in PANELS:
    for m in MUSCLES:
        row = ""
        for p in PROTOCOLS:
            v = shift.get((pan, p, m))
            row += (f"{v['cat']:>12g} mA{v['ano']:>12g} mA{v['pct']:>6.0f} %" if v
                    else f"{'-':>15s}{'-':>15s}{'-':>7s}")
        print(f"{pan:18s}{m:22s}{row}")

### 4b · The response at a matched intensity

Anodic read at the **cathodic threshold** of the same protocol and block, as a % of cathodic
there. **100 % = reversing the field changed nothing.** The cathodic bars sit at 100 % by
construction — they are there so each pair can be seen as a pair.

This is the comparison the thresholds alone cannot make: a polarity can reach threshold at the
same current and still produce a much bigger or smaller response above it.

In [ ]:
warn_if_stale(runs, MT_STAMP,
              used=[(pan, f"{p} · cathodic") for pan in PANELS for p in PROTOCOLS])

tab = condition_table(runs, MT, MUSCLES, CONDITIONS, base=BASE, match="common",
                      group=GROUP, **KW)
fig_vibration(tab, PANELS, MUSCLES, CONDITIONS, colours=COLOUR, hatches=HATCH,
              ylabel="% of cathodic", ylim=(0, 200),   # bars past the top are marked ^ with
                                                       # their real value printed above them
              titles={pan: pan for pan in PANELS},
              title="P05 and P06 — anodic response, % of cathodic at the same intensity")

### The numbers, so they can be checked

In [ ]:
print(f"{'panel':18s}{'muscle':22s}" + "".join(f"{p + ' anodic':>16s}{'mA':>7s}"
                                                                for p in PROTOCOLS))
for pan in PANELS:
    for m in MUSCLES:
        row = ""
        for p in PROTOCOLS:
            v = tab.get((pan, f"{p} · anodic", m), {})
            pct, amp = v.get("pct", float("nan")), v.get("amp")
            row += f"{(f'{pct:.0f} %' if pct == pct else '-'):>16s}{(f'{amp:g}' if amp else '-'):>7s}"
        print(f"{pan:18s}{m:22s}{row}")

### 4c · One figure per muscle — cathodic and anodic, with the EMG

The 1st pulse in mV, the 2nd as a % of it, and the mean of pulses 2–10 as a % of it, with the EMG
underneath in matching colours. The mV panel is the quantity the two percentages are a percentage
**of** — it is all within one participant and one session here, so the millivolts *are* comparable
between the bars.

Along the train is a second, independent readout: two polarities can give the same first response
and behave differently over the following pulses.

In [ ]:
# the intensities the table above settled on, so these figures and that one cannot disagree
MT_cond = {(pan, cond): {m: tab[(pan, cond, m)]["amp"] for m in MUSCLES if (pan, cond, m) in tab}
           for pan in PANELS for cond in CONDITIONS}

for m in MUSCLES:
    fig_bars(runs, MT_cond, m, PANELS, protocols=tuple(CONDITIONS), steps=0,
             colours=COLOUR, hatches=HATCH, names={c: c for c in CONDITIONS},
             min_snr_ratio=2.0, rest_from=2, with_p1=True, traces=True,
             title=m, save=None, **KW)

## 5 · Change a threshold

One cell per recording: move the slider, press **Set to slider**, then **Save**. Re-run section 2
and every figure follows.

> A **cathodic** pick sets the intensity everything downstream is read at, so it changes 4a, 4b
> and 4c. An **anodic** pick changes 4a only — 4b and 4c deliberately read the anodic block at the
> cathodic threshold.

In [ ]:
def repick(key):
    """Open the picker for one recording of this comparison, with a Save button.

    The green line is the DETECTOR's answer for this recording - not another block's, and not
    whatever is already saved (that is the orange trace)."""
    csv = runs[key]
    meta_, t_, sig_ = load_run(csv)
    chans = [c for c in sig_ if c != "Trigger A" and pretty(c) in MUSCLES]
    prev = load_threshold_csv(mt_file(csv), by="channel") if os.path.exists(mt_file(csv)) else None
    detected = motor_thresholds([dict(label="_", csv=csv)], [pretty(c) for c in chans],
                                consecutive=2, verbose=False, **KW)["_"]
    print(f"{key}  {csv.split('/')[-1]}  |  {[m['amp_ma'] for m in meta_]} mA"
          + ("  |  your picks reloaded" if prev else "  |  nothing picked yet"))
    return threshold_picker(meta_, t_, sig_, chans, xlim=(-20, 130), picks=prev, key=csv,
                            suggest=detected, suggest_label="detected",
                            on_save=lambda p_: save_threshold_csv(p_, chans, csv, meta=meta_))

In [ ]:
repick(('P05 · before', '30 Hz · cathodic'))

In [ ]:
repick(('P05 · before', '30 Hz · anodic'))

In [ ]:
repick(('P05 · before', 'ARC-EX · cathodic'))

In [ ]:
repick(('P05 · before', 'ARC-EX · anodic'))

In [ ]:
repick(('P05 · lidocaine', '30 Hz · cathodic'))

In [ ]:
repick(('P05 · lidocaine', '30 Hz · anodic'))

In [ ]:
repick(('P05 · lidocaine', 'ARC-EX · cathodic'))

In [ ]:
repick(('P05 · lidocaine', 'ARC-EX · anodic'))

In [ ]:
repick(('P06 · before', '30 Hz · cathodic'))

In [ ]:
repick(('P06 · before', '30 Hz · anodic'))

In [ ]:
repick(('P06 · before', 'ARC-EX · cathodic'))

In [ ]:
repick(('P06 · before', 'ARC-EX · anodic'))

In [ ]:
repick(('P06 · lidocaine', '30 Hz · cathodic'))

In [ ]:
repick(('P06 · lidocaine', '30 Hz · anodic'))

In [ ]:
repick(('P06 · lidocaine', 'ARC-EX · cathodic'))

In [ ]:
repick(('P06 · lidocaine', 'ARC-EX · anodic'))